# Bank Marketing — Data Preprocessing

## 1. Introduction

### 1.1 Objective & Scope

This notebook establishes a reproducible preprocessing pipeline
for the Bank Marketing classification task.

The main objectives are to:

- Reproduce the established chronological data split.
- Define modeling features and preprocessing groups.
- Establish appropriate numerical and categorical preprocessing strategies.
- Construct a baseline preprocessing pipeline using scikit-learn.
- Validate the transformed feature structure.

The resulting preprocessing configuration will provide the
foundation for baseline modeling and subsequent experiments.

Model training, performance evaluation, and comparisons of
alternative preprocessing strategies are handled in subsequent
notebooks.

### 1.2 Preprocessing Requirements

The preprocessing strategy follows these principles:

**Feature availability**

- Predictions are made immediately before a planned marketing contact.
- `duration` is excluded to prevent data leakage.
- The remaining 19 input variables are retained.

**Data splitting**

- Preserve the chronological 70/20/10 split.
- Fit learned preprocessing on training data only.
- Transform validation data using the fitted preprocessing.
- Keep the test set frozen for final evaluation.

**Feature preprocessing**

- Handle numerical and categorical features separately.
- Preserve `"unknown"` as an explicit category.
- Handle unseen categorical values safely.
- Use a simple baseline representation for `pdays`.
- Avoid unnecessary outlier removal, category merging,
  or feature transformations.

**Reproducibility**

- Use scikit-learn's `ColumnTransformer`.
- Maintain consistent transformations across datasets.
- Keep preprocessing compatible with downstream modeling pipelines.

## 2. Load Data & Splits

This section reproduces the dataset and chronological split
established in `01-data-understanding.ipynb`.

The objectives are to:

- Load the original dataset.
- Reproduce the chronological train/validation/test split.
- Define modeling features and the binary target.
- Verify the consistency of the resulting datasets.

### 2.1 Load Dataset

The original dataset is loaded without modifying its columns.

The `duration` variable remains in the raw dataframe but will
be excluded from modeling features because it is unavailable
at prediction time.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

In [4]:
PROJECT_ROOT = Path.cwd().parent
PROJECT_ROOT

PosixPath('/home/moonk/projects/bank-marketing-ml')

In [5]:
RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "bank-additional"
    / "bank-additional-full.csv"
)

RAW_DATA_PATH

PosixPath('/home/moonk/projects/bank-marketing-ml/data/raw/bank-additional/bank-additional-full.csv')

In [6]:
df = pd.read_csv(RAW_DATA_PATH, sep=";")

In [7]:
df.shape

(41188, 21)

In [8]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


### 2.2 Reproduce Established Splits

The chronological splitting strategy established during EDA
is preserved:

- Training: first 70% (28,831 observations)
- Validation: next 20% (8,238 observations)
- Test: final 10% (4,119 observations)

The original row order is maintained without shuffling.

Training data is used to fit learned preprocessing parameters.

Validation data is transformed using the fitted preprocessing
configuration to verify output consistency.

Model development and validation-based experiments are handled
in subsequent notebooks.

In [9]:
n = len(df)

train_end = int(n * 0.70)
valid_end = int(n * 0.90)

train_df = df.iloc[:train_end].copy()
valid_df = df.iloc[train_end:valid_end].copy()
test_df = df.iloc[valid_end:].copy()

print("Train:", train_df.shape)
print("Validation:", valid_df.shape)
print("Test:", test_df.shape)

Train: (28831, 21)
Validation: (8238, 21)
Test: (4119, 21)


### 2.3 Define Features & Target

The target variable is `y`, converted to binary labels:

- `no` → 0
- `yes` → 1

The modeling feature set excludes `duration`, leaving
19 candidate predictors.

The features and target are separated for training and validation.

Test features and labels will be prepared in the final evaluation
notebook to avoid unnecessary access during model development.

In [60]:
target = "y"
excluded_features = ["duration"]

modeling_features = [
    col
    for col in df.columns
    if col not in [target, *excluded_features]
]

X_train = train_df[modeling_features].copy()
X_valid = valid_df[modeling_features].copy()

y_train = train_df[target].map({"no": 0, "yes": 1})
y_valid = valid_df[target].map({"no": 0, "yes": 1})

### 2.4 Split Validation

Verify that the established split has been reproduced correctly.

The checks cover:

- Expected dataset sizes.
- Non-overlapping chronological boundaries.
- Consistent feature dimensions.
- Successful binary target encoding.

In [61]:
assert len(train_df) == 28_831
assert len(valid_df) == 8_238
assert len(test_df) == 4_119

assert train_df.index.max() < valid_df.index.min()
assert valid_df.index.max() < test_df.index.min()

assert X_train.shape[1] == X_valid.shape[1] == 19
assert "duration" not in X_train.columns

assert y_train.notna().all()
assert y_valid.notna().all()

print("Split and feature validation passed.")

Split and feature validation passed.


### 2.5 Observations

- The established chronological 70/20/10 split was reproduced.
- The resulting datasets contain 28,831 training, 8,238 validation,
  and 4,119 test observations.
- The modeling dataset contains 19 predictors, excluding `duration`.
- The target was converted to binary labels (0 and 1).
- Training and validation datasets are prepared for preprocessing.
- The test set remains frozen for final evaluation.

The next section defines feature groups according to their
preprocessing requirements.

## 3. Feature Definition

This section defines modeling feature groups based on their
characteristics and preprocessing requirements.

The 19 modeling features are divided into three groups:

- **Numerical features:** Standard numerical variables.
- **Categorical features:** Variables requiring categorical encoding.
- **Special features:** Variables requiring domain-specific consideration.

Feature groups are defined explicitly rather than relying solely
on pandas data types.

### 3.1 Define Feature Groups

The `pdays` variable is separated from ordinary numerical features
because its value `999` represents a special sentinel state.

Its representation will be considered separately during
preprocessing strategy development.

In [62]:
numeric_features = [
    "age",
    "campaign",
    "previous",
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "euribor3m",
    "nr.employed",
]

categorical_features = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "day_of_week",
    "poutcome",
]

special_features = [
    "pdays",
]

### 3.2 Validate Feature Groups

Verify that the feature groups:

- Include all 19 modeling features.
- Contain no duplicated features.
- Exclude the target and leakage-related variables.

This ensures that every modeling feature is assigned to
exactly one preprocessing group.

In [63]:
all_features = (
    numeric_features
    + categorical_features
    + special_features
)

assert len(all_features) == len(set(all_features))
assert set(all_features) == set(modeling_features)

print(f"Numerical features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")
print(f"Special features: {len(special_features)}")
print(f"Total modeling features: {len(all_features)}")

Numerical features: 8
Categorical features: 10
Special features: 1
Total modeling features: 19


### 3.3 Feature Definition Summary

The 19 modeling features have been assigned to three
mutually exclusive preprocessing groups:

- 8 numerical features
- 10 categorical features
- 1 special feature (`pdays`)

The feature groups cover all candidate predictors without
duplication or omission.

No features are removed or transformed in this section.

The next section establishes the initial preprocessing strategy
for each feature group.

## 4. Preprocessing Strategy

This section establishes a simple baseline preprocessing strategy
based on the feature characteristics identified during EDA.

The objectives are to:

- Determine appropriate handling of missing and unknown values.
- Define categorical encoding and numerical scaling strategies.
- Determine an initial representation for `pdays`.
- Specify the preprocessing configuration to implement.

Alternative preprocessing strategies will be compared during
subsequent model experiments.

### 4.1 Missing Value Handling

The training data contains no pandas-recognized missing values.

However, several categorical features contain the explicit
category `"unknown"`.

For the baseline preprocessing strategy:

- No missing-value imputation is applied.
- `"unknown"` is retained as an explicit category.
- No observations are removed due to unknown categories.

This preserves the original information without introducing
unnecessary transformations.

Missing-value handling may be revisited if future data
contains actual missing values.

In [64]:
assert not X_train.isna().any().any()

print("No missing values in training features.")

No missing values in training features.


### 4.2 Categorical Encoding

Categorical features are encoded using:

`OneHotEncoder(handle_unknown="ignore")`

One-hot encoding is selected because the categorical features
generally have low cardinality and do not have a consistent
ordinal relationship.

The baseline strategy follows these decisions:

- Retain `"unknown"` as an explicit category.
- Do not manually merge rare categories.
- Do not drop a reference category (`drop="first"`).
- Allow sparse output.
- Handle categories unseen during training without errors.

The encoder learns category mappings from training data only.

Setting `handle_unknown="ignore"` allows unseen categories
in validation or future data to be transformed without errors.
Unseen categories are represented by all zeros within the
corresponding feature's encoded columns.

This provides a simple and reproducible categorical encoding
strategy for the initial baseline.

### 4.3 Numerical Transformation

Numerical features have different units, scales, and
distributional characteristics.

For the Logistic Regression baseline, numerical features
will be standardized using:

`StandardScaler()`

Standardization is useful because:

- Numerical features have substantially different scales.
- Logistic Regression uses regularization that is sensitive
  to feature scaling.
- Comparable feature scales can improve numerical optimization.

The scaler learns the mean and standard deviation from
training data only.

The baseline does not apply:

- Log transformations
- Outlier clipping or winsorization
- Outlier removal
- Correlation-based feature removal

These transformations are not automatically justified by
skewness, extreme values, or high correlations.

Tree-based models generally do not require numerical scaling.
Model-specific preprocessing strategies can be evaluated
during subsequent experiments.

### 4.4 Special Feature: `pdays`

The `pdays` variable requires special consideration because
the value `999` dominates approximately 99.82% of the
training observations.

This value represents a special state rather than an
ordinary continuous day count.

For the initial preprocessing baseline, `pdays` will be:

- Retained in its original numerical representation.
- Standardized together with other numerical features.

This choice prioritizes simplicity and reproducibility.

It does not assume that the raw representation is optimal.

Alternative representations, such as separating the sentinel
state from observed recency values, can be evaluated during
model experimentation.

### 4.5 Baseline Preprocessing Strategy

The initial preprocessing configuration is:

| Feature Group | Preprocessing |
|---|---|
| Numerical features | StandardScaler |
| Categorical features | OneHotEncoder(handle_unknown="ignore") |
| Special feature (`pdays`) | Raw numerical representation with StandardScaler |
| Remaining columns | Drop |

The configuration retains all 19 modeling features.

No additional feature engineering, outlier treatment,
category merging, or correlation-based feature removal
is applied.

All learned preprocessing parameters will be fitted
using training data only.

In [77]:
raw_numeric_features = numeric_features + special_features

assert set(raw_numeric_features + categorical_features) == set(
    modeling_features
)

## 5. Preprocessing Pipeline

This section implements and validates the baseline preprocessing
configuration using scikit-learn.

The objectives are to:

- Construct a reproducible preprocessing pipeline.
- Fit learned transformations on training data only.
- Transform validation data using the fitted preprocessing.
- Verify the transformed feature dimensions and structure.

Model training and predictive performance evaluation
are outside the scope of this notebook.

### 5.1 Build Baseline Preprocessor

The baseline preprocessing configuration uses:

- `StandardScaler` for numerical features, including raw `pdays`.
- `OneHotEncoder(handle_unknown="ignore")` for categorical features.
- `remainder="drop"` for columns not explicitly included.

The transformations are combined using `ColumnTransformer`.

At this stage, the preprocessor is only defined and has not
learned any parameters from the data.

In [78]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            raw_numeric_features,
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
    ],
    remainder="drop",
)

### 5.2 Fit & Transform

The preprocessor is fitted exclusively on the training data.

During fitting:

- `StandardScaler` learns numerical means and standard deviations.
- `OneHotEncoder` learns categorical levels observed in training.

The fitted preprocessor is then applied to validation data
without learning new parameters.

This ensures consistent preprocessing while preventing
data leakage from validation into training.

In [79]:
X_train_prepared = preprocessor.fit_transform(X_train)

X_valid_prepared = preprocessor.transform(X_valid)

print("Train:", X_train_prepared.shape)
print("Validation:", X_valid_prepared.shape)

Train: (28831, 61)
Validation: (8238, 61)


### 5.3 Validate Preprocessing Output

The transformed datasets are checked for:

- Consistent observation counts.
- Identical transformed feature dimensions.
- Successful retrieval of transformed feature names.
- Exclusion of the target and leakage-related variables.

The validation data must use the same feature representation
learned from the training data.

In [80]:
feature_names = preprocessor.get_feature_names_out()

assert X_train_prepared.shape[0] == len(X_train)
assert X_valid_prepared.shape[0] == len(X_valid)

assert X_train_prepared.shape[1] == X_valid_prepared.shape[1]
assert X_train_prepared.shape[1] == len(feature_names)

assert "duration" not in preprocessor.feature_names_in_
assert target not in preprocessor.feature_names_in_

print(f"Transformed features: {len(feature_names)}")
print("Preprocessing validation passed.")

Transformed features: 61
Preprocessing validation passed.


In [81]:
pd.Series(feature_names, name="feature_name").to_frame()

,feature_name
0,num__age
1,num__campaign
2,num__previous
3,num__emp.var.rate
4,num__cons.price.idx
...,...
56,cat__day_of_week_tue
57,cat__day_of_week_wed
58,cat__poutcome_failure
59,cat__poutcome_nonexistent


### 5.4 Observations

- The baseline preprocessing configuration was implemented
  using scikit-learn's `ColumnTransformer`.
- Numerical features, including raw `pdays`, were standardized.
- Categorical features were one-hot encoded.
- All learned preprocessing parameters were fitted on training
  data only.
- Validation data was transformed using the fitted preprocessor.
- The transformed training and validation datasets have
  consistent feature dimensions.
- The resulting preprocessing configuration can be integrated
  into a complete modeling pipeline.

No predictive model was trained or evaluated in this notebook.

## 6. Preprocessing Summary

This notebook established a reproducible baseline preprocessing
configuration for the Bank Marketing classification task.

The preprocessing strategy was designed based on feature
characteristics, prediction-time availability, and leakage prevention.

### 6.1 Baseline Preprocessing Configuration

The initial preprocessing configuration is:

| Component | Configuration |
|---|---|
| Numerical features | StandardScaler |
| Categorical features | OneHotEncoder(handle_unknown="ignore") |
| Special feature (`pdays`) | Raw numerical representation |
| Remaining columns | Drop |

The configuration retains all 19 modeling features,
excluding `duration`.

Key decisions:

- Retain `"unknown"` as an explicit category.
- Handle unseen categorical values safely.
- Preserve all categorical levels without dropping a reference category.
- Do not manually merge rare categories.
- Do not apply outlier removal, clipping, or additional numerical transformations.
- Retain correlated macroeconomic features.

The configuration prioritizes simplicity and reproducibility
rather than attempting to optimize preprocessing before
establishing a baseline model.

### 6.2 Preprocessing Validation

The preprocessing configuration was implemented using
scikit-learn's `ColumnTransformer`.

- Learned preprocessing parameters were fitted on training data only.
- Validation data was transformed without refitting.
- Transformed feature dimensions were checked for consistency.
- Feature names were successfully retrieved.
- The target and excluded features were not included in the
  preprocessing input.

The resulting preprocessor is ready for integration into
a scikit-learn modeling pipeline.

### 6.3 Design Decisions & Limitations

**Data leakage prevention**

- `duration` is excluded from modeling features.
- The chronological 70/20/10 split is preserved.
- Learned preprocessing uses training data only.
- The test set remains frozen for final evaluation.

**Known limitations**

- Standardization is suitable for the planned linear baseline,
  but may be unnecessary for tree-based models.
- Raw `pdays` contains a sentinel value that does not represent
  ordinary elapsed time.
- Rare categories and extreme numerical values are retained.
- Correlated macroeconomic variables may affect linear-model
  coefficient interpretation.
- Temporal distribution shift may affect model generalization.

These limitations do not automatically justify additional
preprocessing.

Alternative strategies should be evaluated through controlled
validation experiments.

### 6.4 Next Steps

The next notebook, `03_baseline_modeling.ipynb`, will:

1. Reconstruct the baseline preprocessing configuration.
2. Establish and train a Logistic Regression baseline.
3. Evaluate validation Average Precision and ROC-AUC.
4. Evaluate business-oriented ranking metrics:
   Precision@K, Recall@K, and Lift@K.
5. Document baseline results and limitations.

Subsequently, `04_model_experiments.ipynb` will compare alternative
feature representations, preprocessing strategies, and model families.

The selected baseline will serve as the reference for
these experiments.

The test set will not be used for model selection,
hyperparameter tuning, or ranking-policy selection.